# 02 - Evaluate Random Forest on Held-out Sparse N-gram Features

Loads the model trained in `01_train_rf.ipynb`, loads/stacks the chunked
sparse test set, validates the feature dimensionality, runs predictions,
and writes evaluation artifacts (`test_predictions.csv`, `metrics.json`).


In [5]:
# ---------------------------------------------------------------------------
# Cell 1: Setup, imports, configuration
# ---------------------------------------------------------------------------
import os
import gc
import glob
import json
import joblib
import numpy as np
import scipy.sparse as sp
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

# ---- Configuration ----
NGRAM_COUNT = 4  # must match the value used in 01_train_rf.ipynb
TEST_DIR = os.path.join("feature_datasets", f"{NGRAM_COUNT}gram", "test")    # directory holding X_batch_iii.npz / y_batch_iii.npy
OUTPUT_DIR = os.path.join("outputs", f"{NGRAM_COUNT}gram")
# Auto-detect chunk count from the directory instead of hardcoding it --
# test/ is not guaranteed to have the same number of chunks as train/.
N_CHUNKS = len(glob.glob(os.path.join(TEST_DIR, "X_batch_*.npz")))
if N_CHUNKS == 0:
    raise FileNotFoundError(f"No X_batch_*.npz files found in {TEST_DIR}")

# Create the nested output directory if it doesn't already exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"NGRAM_COUNT = {NGRAM_COUNT}")
print(f"TEST_DIR    = {TEST_DIR}")
print(f"OUTPUT_DIR  = {OUTPUT_DIR}")
print(f"N_CHUNKS    = {N_CHUNKS} (auto-detected)")


NGRAM_COUNT = 4
TEST_DIR    = feature_datasets\4gram\test
OUTPUT_DIR  = outputs\4gram
N_CHUNKS    = 9 (auto-detected)


In [6]:
# ---------------------------------------------------------------------------
# Cell 2: Load the trained model and load/stack the test chunks
# ---------------------------------------------------------------------------
def load_chunks(data_dir, n_chunks, prefix_features="X_batch", prefix_labels="y_batch"):
    """
    Memory-efficient loader for sparse feature chunks + label chunks.
    Mirrors the loader in 01_train_rf.ipynb; kept local here so this
    notebook can be run standalone.

    Returns
    -------
    X : scipy.sparse.csr_matrix, shape (n_samples, n_features)
    y : np.ndarray, shape (n_samples,)
    """
    X_parts = []
    y_parts = []

    for i in range(n_chunks):
        x_path = os.path.join(data_dir, f"{prefix_features}_{i:03d}.npz")
        y_path = os.path.join(data_dir, f"{prefix_labels}_{i:03d}.npy")

        if not os.path.exists(x_path):
            raise FileNotFoundError(f"Missing feature chunk: {x_path}")
        if not os.path.exists(y_path):
            raise FileNotFoundError(f"Missing label chunk: {y_path}")

        X_chunk = sp.load_npz(x_path).tocsr()
        y_chunk = np.load(y_path)

        if X_chunk.shape[0] != y_chunk.shape[0]:
            raise ValueError(
                f"Row mismatch in chunk {i}: "
                f"X has {X_chunk.shape[0]} rows, y has {y_chunk.shape[0]}"
            )

        X_parts.append(X_chunk)
        y_parts.append(y_chunk)
        print(f"[{i + 1}/{n_chunks}] loaded {x_path}  shape={X_chunk.shape}")

        if (i + 1) % 6 == 0:
            gc.collect()

    print("Stacking chunks with scipy.sparse.vstack / np.concatenate ...")
    X = sp.vstack(X_parts, format="csr")
    y = np.concatenate(y_parts)
    del X_parts, y_parts
    gc.collect()

    print(f"Final stacked matrix: X.shape={X.shape}, y.shape={y.shape}")
    return X, y


# --- Load trained model ---
model_path = os.path.join(OUTPUT_DIR, "rf_model.joblib")
if not os.path.exists(model_path):
    raise FileNotFoundError(
        f"Trained model not found at {model_path}. Run 01_train_rf.ipynb first."
    )

rf = joblib.load(model_path)
print(f"Loaded model from: {model_path}")
print(f"Model expects n_features_in_ = {rf.n_features_in_}")

# --- Load & stack test chunks ---
X_test, y_true = load_chunks(TEST_DIR, N_CHUNKS)


Loaded model from: outputs\4gram\rf_model.joblib
Model expects n_features_in_ = 1124274
[1/9] loaded feature_datasets\4gram\test\X_batch_000.npz  shape=(100, 1124274)
[2/9] loaded feature_datasets\4gram\test\X_batch_001.npz  shape=(100, 1124274)
[3/9] loaded feature_datasets\4gram\test\X_batch_002.npz  shape=(100, 1124274)
[4/9] loaded feature_datasets\4gram\test\X_batch_003.npz  shape=(100, 1124274)
[5/9] loaded feature_datasets\4gram\test\X_batch_004.npz  shape=(100, 1124274)
[6/9] loaded feature_datasets\4gram\test\X_batch_005.npz  shape=(100, 1124274)
[7/9] loaded feature_datasets\4gram\test\X_batch_006.npz  shape=(100, 1124274)
[8/9] loaded feature_datasets\4gram\test\X_batch_007.npz  shape=(100, 1124274)
[9/9] loaded feature_datasets\4gram\test\X_batch_008.npz  shape=(81, 1124274)
Stacking chunks with scipy.sparse.vstack / np.concatenate ...
Final stacked matrix: X.shape=(881, 1124274), y.shape=(881,)


In [7]:
# ---------------------------------------------------------------------------
# Cell 3: Sanity check - test feature dimensionality must match the model
# ---------------------------------------------------------------------------
assert X_test.shape[1] == rf.n_features_in_, (
    f"Feature mismatch: test data has {X_test.shape[1]} features, "
    f"but the model was trained on {rf.n_features_in_} features. "
    f"Check that NGRAM_COUNT ({NGRAM_COUNT}) matches the training run "
    f"that produced outputs/ngram_{NGRAM_COUNT}/rf_model.joblib."
)
print(
    "Feature dimension check passed: "
    f"X_test.shape[1] == rf.n_features_in_ == {rf.n_features_in_}"
)


Feature dimension check passed: X_test.shape[1] == rf.n_features_in_ == 1124274


In [8]:
# ---------------------------------------------------------------------------
# Cell 4: Predict, evaluate, and save results
# ---------------------------------------------------------------------------
# Predict directly on the sparse CSR matrix (no densification)
print("Running predictions ...")
y_pred = rf.predict(X_test)

# --- Evaluation metrics (raw, 0-1 scale, full precision) ---
acc = accuracy_score(y_true, y_pred)
report_dict = classification_report(y_true, y_pred, output_dict=True)


def to_percent_report(report_dict, digits=4):
    """
    Convert a sklearn classification_report(output_dict=True) dict to a
    percentage scale (0-100), rounded to `digits` decimal places, for
    precision/recall/f1-score. 'support' is a raw sample count, not a
    rate, so it is left unrounded/unconverted.
    """
    percent_report = {}
    for key, val in report_dict.items():
        if isinstance(val, dict):
            # per-class rows and "macro avg" / "weighted avg" rows
            percent_report[key] = {
                "precision": round(val["precision"] * 100, digits),
                "recall": round(val["recall"] * 100, digits),
                "f1-score": round(val["f1-score"] * 100, digits),
                "support": val["support"],
            }
        else:
            # top-level "accuracy" scalar
            percent_report[key] = round(val * 100, digits)
    return percent_report


report_dict_percent = to_percent_report(report_dict)

# --- Print a percentage-formatted table (whole number + 4 decimal places) ---
print(f"Accuracy: {report_dict_percent['accuracy']:.4f}%")
print()
header = f"{'':<14}{'precision':>12}{'recall':>12}{'f1-score':>12}{'support':>10}"
print(header)
for label, vals in report_dict_percent.items():
    if label == "accuracy":
        continue
    row = (
        f"{label:<14}"
        f"{vals['precision']:>11.4f}%"
        f"{vals['recall']:>11.4f}%"
        f"{vals['f1-score']:>11.4f}%"
        f"{vals['support']:>10}"
    )
    print(row)

# --- a. Save predictions CSV ---
preds_df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred})
preds_path = os.path.join(OUTPUT_DIR, "test_predictions.csv")
preds_df.to_csv(preds_path, index=False)
print(f"\nPredictions saved to: {preds_path}")

# --- b. Save metrics JSON ---
# Both raw (0-1, full precision) and percent (0-100, 4 decimal places)
# versions are saved -- raw for anything downstream that expects
# standard sklearn-scale metrics, percent to match the console output above.
metrics = {
    "ngram_count": NGRAM_COUNT,
    "n_test_samples": int(X_test.shape[0]),
    "n_features": int(X_test.shape[1]),
    "accuracy": acc,
    "accuracy_percent": report_dict_percent["accuracy"],
    "classification_report": report_dict,
    "classification_report_percent": report_dict_percent,
}
metrics_path = os.path.join(OUTPUT_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to: {metrics_path}")

# Cleanup
del X_test
gc.collect()


Running predictions ...
Accuracy: 98.4109%

                 precision      recall    f1-score   support
0                 99.0276%    98.7076%    98.8673%     619.0
1                 96.9697%    97.7099%    97.3384%     262.0
macro avg         97.9986%    98.2088%    98.1029%     881.0
weighted avg      98.4156%    98.4109%    98.4126%     881.0

Predictions saved to: outputs\4gram\test_predictions.csv
Metrics saved to: outputs\4gram\metrics.json


57